# 🌍 TripForge: Autonomous Multi-Agent Travel Concierge
### Interactive Demonstration, Architecture Pitch, & Code Submission Notebook

---

## 📖 1. The Pitch: Problem, Solution, and Value

### 1.1 Problem Statement
Planning travel is notoriously time-consuming and fragile. Travel planners face three core challenges:
1. **Cognitive Overload**: Sifting through thousands of activities, restaurants, transit options, and visa requirements while keeping track of traveler budgets, accessibility needs, and dietary restrictions.
2. **Itinerary Fragility**: Travel plans are static. A sudden transit strike, museum closure, or localized extreme weather event can ruin a carefully planned day, forcing manual, stressful replanning on the fly.
3. **Data Privacy Risks**: Inputting sensitive travel preferences, group details, and PII into various third-party cloud engines exposes personal data to leakages and unconsented tracking.

### 1.2 The Solution
**TripForge** is an autonomous, multi-agent travel concierge system built using **Google's Agent Development Kit (ADK)** and the **Model Context Protocol (MCP)**. It dynamically designs hyper-personalized, day-by-day travel itineraries and handles real-time disruptions automatically in the browser.

Key components of the solution include:
- **Multi-Agent Orchestration**: A pipeline of four specialized ADK agents (Profile, Research, Itinerary, Disruption) that sequentially refine raw preferences into a final schedule.
- **Local-First MCP Server**: A standalone FastMCP server running local travel research databases, weather estimators, and active disruption checkers.
- **Security-by-Design**: Traveler profiles are encrypted locally using hardware-derived Fernet cryptography, PII is automatically redacted from logs, and local tools require verified HMAC-SHA256 signatures to execute.
- **Resilient Production Design**: Built to run on read-only environments (e.g. serverless containers) and fully compliant with Python 3.12 regex specifications.

---

## 🏗️ 2. System Architecture & Agent Codebase

TripForge uses a sequential pipeline pattern linked to a standalone MCP server, pushing live progress and log updates to a beautiful Flask dashboard via Server-Sent Events (SSE).

```
               ┌────────────────────────┐
               │    Flask Web Server    │
               │        (app.py)        │
               └───────────┬────────────┘
                           │ (SSE Live Progress Stream)
                           ▼
               ┌────────────────────────┐
               │ TripForge Orchestrator │
               └───────────┬────────────┘
                           │
         ┌─────────────────┼─────────────────┐
         ▼                 ▼                 ▼
   ProfileAgent  ──► ResearchAgent  ──► ItineraryAgent  ──► DisruptionAgent
  (Validate/Audit)  (Fetch MCP Info)   (Design Schedule)  (Check Disruption)
         │                 │                 │                 │
         └─────────────────┼─────────────────┴─────────────────┘
                           ▼ (HMAC Signed StdIO Calls)
               ┌────────────────────────┐
               │  FastMCP Travel Tools  │
               │     (MCP Subprocess)   │
               ├────────────────────────┤
               │ - Weather Tool         │
               │ - Activities Search    │
               │ - Country Info Check   │
               │ - Transport Estimator  │
               │ - Disruption Checker   │
               └────────────────────────┘
```

### 2.1 The Agents We Use (Code Definitions)

Here is the complete source code for each of the four ADK agents imported and run in the pipeline:

#### 👤 Profile Agent (`tripforge/agents/profile_agent.py`)
```python
from google.adk.agents import Agent

def get_profile_agent(model_name: str = "gemini-2.5-flash") -> Agent:
    instruction = (
        "You are TripForge's Profile Manager. Your job is to take raw traveler preferences "
        "and create a structured, validated traveler profile. "
        "You must analyze the inputs and produce a structured JSON object representing the "
        "traveler profile, followed by a human-readable confirmation summary. "
        "Specifically, verify the following fields and handle them as follows:\n"
        "- destination: Validate and clean the destination name. Format to standard title casing (e.g. 'Paris').\n"
        "- days: Must be between 1 and 30. If missing, default to 7.\n"
        "- travelers: Must be at least 1. If missing, default to 2.\n"
        "- budget: Must be a positive number. If missing, default to 1500.0.\n"
        "- currency: Validate currency string (e.g. USD, EUR). Default to USD.\n"
        "- start_date: Check format YYYY-MM-DD. If missing, default to upcoming date.\n"
        "- accessibility_needs: Explicitly check accessibility constraints. If not specified, set to None.\n"
        "- dietary_restrictions: Explicitly check dietary requirements. If not specified, set to None.\n"
        "- travel_style: If not specified, infer based on budget per person per day (> $300 luxury, $100-$300 mid-range, < $100 budget).\n"
        "- interests: List of travel interests. If missing, default to ['general-sightseeing'].\n\n"
        "Safety Check: You must highlight accessibility and dietary needs explicitly, confirming them as safety-critical.\n\n"
        "Your final response must be formatted as:\n"
        "---PROFILE_JSON---\n"
        "[Insert clean, valid JSON block containing all keys]\n"
        "---SUMMARY---\n"
        "[Insert a concise, friendly summary of the travelers, their style, budget, safety needs, and interests]"
    )
    return Agent(
        name="ProfileAgent",
        model=model_name,
        description="Validates and enriches traveler preferences into a structured, safety-checked traveler profile.",
        instruction=instruction,
        tools=[]
    )
```

#### 🔍 Research Agent (`tripforge/agents/research_agent.py`)
```python
from typing import List, Any
from google.adk.agents import Agent

def get_research_agent(model_name: str = "gemini-2.5-flash", tools: List[Any] = None) -> Agent:
    instruction = (
        "You are TripForge's Research Specialist. Given a validated traveler profile, "
        "your goal is to gather comprehensive destination intelligence using the provided tools.\n\n"
        "You MUST execute the following steps:\n"
        "1. Check the weather forecast using get_weather for EACH day of the planned trip duration starting from start_date.\n"
        "2. Retrieve country travel essentials using get_country_info for the destination country.\n"
        "3. Search for available activities using search_activities. Call it multiple times with different categories "
        "to build a wide, diverse pool of options. Respect accessibility and dietary filters.\n"
        "4. Estimate the transport costs between the traveler's origin and destination using estimate_transport_cost.\n"
        "5. Compile all these findings into a structured research report.\n\n"
        "The report must contain: Weather forecast, Activity pool of 15-20 options, Country essentials, Transport estimates, and safety notes.\n\n"
        "Your final response must be formatted clearly with sections, starting with: '---RESEARCH_REPORT---'"
    )
    return Agent(
        name="ResearchAgent",
        model=model_name,
        description="Researches weather, activities, transit, and country specifics using MCP tools.",
        instruction=instruction,
        tools=tools or []
    )
```

#### 📅 Itinerary Agent (`tripforge/agents/itinerary_agent.py`)
```python
from typing import List, Any
from google.adk.agents import Agent

def get_itinerary_agent(model_name: str = "gemini-2.5-flash", tools: List[Any] = None) -> Agent:
    instruction = (
        "You are TripForge's Master Itinerary Architect. Your job is to take a traveler profile "
        "and a research report, and build a perfect day-by-day travel plan.\n\n"
        "You must check all planned activities for disruptions using check_disruption. "
        "If you need more activities to fill slots, use search_activities.\n\n"
        "Rules:\n"
        "1. Never schedule outdoor activities on bad-weather days.\n"
        "2. Respect accessibility requirements on EVERY activity.\n"
        "3. Keep daily costs within the per-day budget.\n"
        "4. Balance intensity (max two high-energy activities/day).\n"
        "5. Account for travel time between activities (min 30 min buffer).\n"
        "6. Include dining suggestions respecting dietary restrictions.\n"
        "7. End each day with a relaxing evening activity.\n"
        "8. Include a 'hidden gem' non-tourist activity.\n"
        "9. Ensure no activities are disrupted according to check_disruption.\n"
        "10. Generate detailed sections and schedule slots for EVERY single day requested. No truncation.\n\n"
        "Your final response must contain a structured Markdown itinerary followed by a structured JSON block:\n\n"
        "---ITINERARY_MARKDOWN---\n"
        "[Insert beautiful markdown itinerary here]\n\n"
        "---ITINERARY_JSON---\n"
        "[Insert clean, valid JSON block representing the structured data with keys: destination, days, total_cost, days_list]"
    )
    return Agent(
        name="ItineraryAgent",
        model=model_name,
        description="Constructs a balanced day-by-day travel plan conforming to weather, budget, accessibility, and dining constraints.",
        instruction=instruction,
        tools=tools or []
    )
```

#### ⚡ Disruption Agent (`tripforge/agents/disruption_agent.py`)
```python
from typing import List, Any
from google.adk.agents import Agent

def get_disruption_agent(model_name: str = "gemini-2.5-flash", tools: List[Any] = None) -> Agent:
    instruction = (
        "You are TripForge's Disruption Response Specialist. "
        "When given a disruption event, an existing itinerary, and a traveler profile, you must replan the itinerary autonomously.\n\n"
        "Follow these steps:\n"
        "1. Identify exactly which activities are affected by the disruption.\n"
        "2. Call check_disruption to verify disruption details, severity, and timing.\n"
        "3. Find suitable replacement activities using search_activities.\n"
        "4. Call get_weather to verify weather conditions for the replacement day.\n"
        "5. Call check_disruption on replacement activities to ensure they are safe.\n"
        "6. Preserve as much of the original itinerary as possible.\n"
        "7. Maintain the overall budget balance.\n"
        "8. Produce the final updated itinerary with changes highlighted using the ⚡ emoji.\n"
        "9. Add a clear, brief 'What Changed' summary section at the very top.\n"
        "10. Generate detailed sections and slots for EVERY single day requested. No placeholders.\n\n"
        "Your final response must be formatted as:\n\n"
        "---REPLANNED_MARKDOWN---\n"
        "[Insert updated markdown itinerary with ⚡ emojis]\n\n"
        "---REPLANNED_JSON---\n"
        "[Insert clean, valid JSON block representing the updated structured data]"
    )
    return Agent(
        name="DisruptionAgent",
        model=model_name,
        description="Analyzes disruptions, finds alternative activities, and dynamically reconstructs itineraries.",
        tools=tools or []
    )
```

### 2.2 Custom Workspace Agent Skill (`.agents/skills/tripforge_manager/SKILL.md`)

TripForge uses custom agent workspace skills (customizations loaded by the Antigravity developer agent) to manage environments and execute tasks correctly. Below is the complete codebase content of the custom developer agent skill loaded at `.agents/skills/tripforge_manager/SKILL.md`:

```markdown
---
name: "tripforge_manager"
description: "Triggers when editing, testing, debugging, or managing the TripForge travel concierge system."
---

# TripForge Manager Skill

This skill provides context and guidelines for managing, testing, and developing the TripForge travel planner.

## Architectural Core
TripForge links Google's Agent Development Kit (ADK) with a Model Context Protocol (MCP) server:
- **Flask Server**: App entrypoint at app.py.
- **Multi-Agent Orchestrator**: Async pipeline at tripforge/orchestrator.py.
- **MCP Travel Server**: Standalone subprocess at travel_tools_server.py.
- **Security & Encryption**: Utilities at security.py.

---

## Development & Debugging Rules

### 1. Compile Check
Before completing any changes, always run a compilation verification check to ensure no syntax errors are present:
```bash
python -m py_compile app.py tripforge/orchestrator.py
```

### 2. Subprocess PYTHONPATH Propagation
When launching the MCP subprocess via StdioServerParameters, the child process must inherit all parent process import search paths to resolve third-party packages (like httpx). Always combine base_dir and the parent's sys.path and set it as PYTHONPATH using the platform-specific separator:
```python
python_path_dirs = [base_dir] + [p for p in sys.path if p]
```

### 3. Read-only Filesystem Compatibility (e.g. Vercel)
Vercel/Lambda filesystems are read-only except for /tmp.
- **Logs**: If writing to tripforge.log raises an OSError with errno == 30 (read-only filesystem), silently ignore the write failure. Logs are already outputted to sys.stderr and captured automatically by the hosting platform console.
- **Downloads**: If creating or writing to the local /scratch directory fails, fall back to the system's temporary directory (tempfile.gettempdir()).

### 4. Regex Parsing Boundaries
Never use $$ for end-of-string matching inside Python regexes. Python 3.12 evaluates $$ as a zero-width match at every position in the string, which causes non-greedy searches to match empty strings. Use a single $ for end-of-string matching, and employ lenient patterns to match structured outputs reliably.
```

### 2.3 The Agent Skills We Use
The multi-agent system is empowered by several custom agent skills that drive autonomy and safety:
- **Dynamic Tool Invocation & Routing**: The agents possess the skill to dynamically discover, parameterize, and call custom FastMCP protocol tools (e.g. querying country-specific tipping customs or estimating carbon emissions of transit options).
- **Contextual Self-Correction & Replanning**: The DisruptionAgent detects scheduled segments affected by closures and uses search databases to substitute the affected slots while keeping the total budget, travel buffers, and traveler constraints balanced.
- **Real-Time Asynchronous Streaming**: Using asynchronous Python generator patterns, the system streams the step-by-step progress and raw reasoning tokens of each agent directly back to the Flask dashboard to give users a fully transparent, vibecoding experience.
- **Cryptographic Auditing & Log scrubbing**: Safety skills built into the pipeline automatically redact sensitive PII (emails, phone numbers, passport digits) from all logs and require cryptographic tool signatures to prevent unauthorized prompt injection tool execution.

## ⚙️ 3. Environment Setup & Dependency Verification

Let's import critical modules, verify project paths, and confirm that all required packages are present.

In [ ]:
import os
import sys
import json
import asyncio
from datetime import datetime
from dotenv import load_dotenv

# Ensure project root is in the Python search path
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"[*] Project root directory: {project_root}")
print(f"[*] Python Executable: {sys.executable}")

# Load environment variables from .env
load_dotenv()

# Check core dependencies
try:
    import google.adk
    import mcp
    import cryptography
    import flask
    import rich
    print("[+] Core dependencies imported successfully!")
except ImportError as e:
    print(f"[-] Missing dependency: {e}. Please run 'pip install -r requirements.txt'")

## 🔒 4. Key Concept: Security & Privacy Features (Code Demo)

To ensure traveler data is safe, TripForge implements a four-tiered security system:
1. **Profile Data Encryption**: Traveler preferences are encrypted locally using symmetric Fernet cryptography with a dynamically derived hardware fingerprint (no key saved on disk).
2. **PII Scrubbing**: Logs are automatically scrubbed using regex rules to replace emails, phone numbers, and passport-like digits with `[REDACTED]` tokens.
3. **Input Sanitization**: Input fields are validated against strict filters. Destinations are checked against supported cities, and budgets are limited to a safe range to block injection patterns.
4. **MCP Call Signing**: MCP tool calls are signed with an HMAC-SHA256 signature containing parameters and verified by the server before tool execution.

Let's execute code demonstrations of these security controls below:

In [ ]:
from tripforge.utils.security import encrypt_profile, decrypt_profile_content, get_machine_key, scrub_pii, sign_tool_call, verify_tool_call

# ----------------- 4.1 Symmetric Profile Encryption -----------------
mock_profile = {
    "destination": "Paris",
    "days": 3,
    "travelers": 2,
    "budget": 2000.0,
    "currency": "USD",
    "accessibility_needs": "None",
    "dietary_restrictions": "Gluten-Free"
}

machine_key = get_machine_key()
print(f"[*] 4.1.1 Derived Machine Key (Base64): {machine_key.decode()[:20]}...")

encrypted_data = encrypt_profile(mock_profile)
print(f"[*] 4.1.2 Encrypted Profile Bytes: {encrypted_data[:40]}...")

decrypted_profile = decrypt_profile_content(encrypted_data)
print(f"[+] 4.1.3 Successfully Decrypted Profile JSON:")
print(json.dumps(decrypted_profile, indent=2))

# ----------------- 4.2 PII Log Scrubbing -----------------
raw_log_entry = (
    "Booking flight for John Doe (Passport: US987654321, Phone: +1-555-0199-823) "
    "and confirmation sent to john.doe@example.com."
)
scrubbed_log = scrub_pii(raw_log_entry)
print("\n[*] 4.2 PII Scrubbing Output:")
print(f"Raw log:      {raw_log_entry}")
print(f"Scrubbed log: {scrubbed_log}")

# ----------------- 4.3 HMAC Tool Call Signing -----------------
tool_name = "get_weather"
tool_params = {"city": "Tokyo", "date": "2026-07-15"}

signature = sign_tool_call(tool_name, tool_params)
print(f"\n[*] 4.3.1 Generated HMAC Signature: {signature}")

is_valid = verify_tool_call(tool_name, tool_params, signature)
print(f"[+] 4.3.2 Verification check with original parameters: {is_valid}")

tampered_params = {"city": "Tokyo", "date": "2026-07-15", "extra_injected_cmd": "rm -rf /"}
is_tampered_valid = verify_tool_call(tool_name, tampered_params, signature)
print(f"[-] 4.3.3 Verification check with tampered parameters: {is_tampered_valid} (Should be False)")

## 🔌 5. Key Concept: Model Context Protocol (MCP) Server

TripForge uses a custom python FastMCP server located at `tripforge/mcp_server/travel_tools_server.py` to provide real-time external data to the agents.

The server exposes the following tools:
- `get_weather(city, date)`: Fetches live temperature and conditions using latitude/longitude lookups.
- `search_activities(city, accessibility_required, dietary_preference, category)`: Scans the local SQLite/JSON database to select accessible and diet-safe options.
- `get_country_info(country_name)`: Returns currencies, timezone, local emergency digits, visa rules, and tipping etiquettes.
- `estimate_transport_cost(origin_city, destination_city, transport_type)`: Simulates travel fares, travel duration, and carbon emissions.
- `check_disruption(city, activity_name, date)`: Resolves active museum closures, transit strikes, or storm-level shutdowns.

Let's call the tool handlers directly from the python module to verify their outputs:

In [ ]:
from tripforge.mcp_server.travel_tools_server import get_weather, get_country_info, check_disruption

# Call weather tool
weather_res = await get_weather(city="Paris", date="2026-08-15")
print("--- Weather Tool Response ---")
print(json.dumps(weather_res, indent=2))

# Call country info tool
country_res = await get_country_info(country_name="France")
print("\n--- Country Info Tool Response ---")
print(json.dumps(country_res, indent=2))

# Call disruption checker tool
disruption_res = await check_disruption(city="Paris", activity_name="Louvre Museum", date="2026-06-27")
print("\n--- Disruption Check Tool Response ---")
print(json.dumps(disruption_res, indent=2))

## 🤖 6. Key Concept: Live Multi-Agent Orchestration (Google ADK & Agent Skills)

Below we execute the **live** multi-agent pipeline using Google's code-first `google-adk` framework. 
This runs the actual sequential planner using the Gemini models configured in the environment. Make sure your `GOOGLE_API_KEY` is present in the `.env` file or environment variables before executing.

In [ ]:
from tripforge.orchestrator import stream_tripforge

async def run_live_planning_simulation():
    if not os.getenv("GOOGLE_API_KEY"):
        print("[-] ERROR: GOOGLE_API_KEY is not set in the environment.")
        print("[*] Please set your Gemini API key in the environment or in the .env file to run this live agent simulation.")
        return
        
    print("[*] Launching live multi-agent pipeline stream using Google ADK...")
    
    # Force live mode
    os.environ["TRIPFORGE_MODE"] = "live"
    
    generator = stream_tripforge(
        destination="Paris",
        days=3,
        travelers=2,
        budget=2000.0,
        currency="USD",
        interests=["culture", "food", "history"]
    )
    
    async for event in generator:
        event_type = event.get("type")
        if event_type == "progress":
            step = event.get("step")
            msg = event.get("message")
            icon = event.get("icon", "🤖")
            print(f"\n{icon} [Step {step}/4] {msg}")
        elif event_type == "agent_start":
            print(f"    -> Agent {event.get('agent')} started reasoning...")
        elif event_type == "agent_stream":
            content = event.get("content", "")
            print(f"\033[92m{content}\033[0m", end="", flush=True)
        elif event_type == "complete":
            print("\n\n[+] Multi-Agent Planning Completed Successfully!")
            print(f"Summary Weather Conditions: {event.get('summary', {}).get('weather_condition')}")
            print(f"Total Calculated Spend: {event.get('summary', {}).get('currency')} {event.get('summary', {}).get('total_cost')}")
            
            itinerary = event.get("itinerary", "")
            print("\n--- Generated Itinerary ---")
            print(itinerary)

# Run the async simulator
await run_live_planning_simulation()

## 🌐 7. Key Concept: Deployability & Production Resilience

### 7.1 Read-Only Filesystem Adaptation
Serverless hosts (like Vercel or AWS Lambda) mount filesystems as read-only. TripForge avoids crashes due to write limit violations by:
- Catching `OSError` with `errno == 30` (read-only filesystem) when trying to write log files and gracefully falling back to standard terminal output.
- Redirecting storage downloads from the default workspace folder to `tempfile.gettempdir()` if local directory creation fails.

Let's run a simulation of our logger catching read-only constraints:

In [ ]:
import errno

def log_error_simulation(error: Exception):
    try:
        # Simulate trying to write to a read-only directory
        raise OSError(errno.EROFS, "Read-only file system")
    except OSError as e:
        if e.errno == 30:
            print("[+] Caught Read-Only Filesystem error (errno 30) - Redirecting logs to sys.stderr.")
        else:
            print(f"[-] Unhandled OS error: {e}")
    except Exception as e:
        print(f"[-] Other error: {e}")

log_error_simulation(OSError(30, "Read-only file system"))

### 7.2 Python 3.12 Regex Boundary Safety
Python 3.12 introduces stricter constraints for regular expressions. Using double dollar signs `$$` inside search patterns is evaluated as a zero-width match at *every* location, causing parsing loops. 
TripForge uses single `$` anchors for end-of-string boundaries to guarantee compatibility:

In [ ]:
import re

pattern_safe = re.compile(r"---PROFILE_JSON---(.*)---SUMMARY---$", re.DOTALL)
sample_text = "---PROFILE_JSON---\n{\n  \"destination\": \"Paris\"\n}\n---SUMMARY---"

match = pattern_safe.search(sample_text)
if match:
    print("[+] Successfully parsed sections using Python 3.12 compliant boundary pattern!")
    print(f"Extracted Profile: {match.group(1).strip()}")

## 🛸 8. Container Deployment to Google Cloud Run

To make **TripForge** accessible publicly on the web, we containerize and deploy it to **Google Cloud Run**, a serverless container hosting platform. Google Cloud Run matches our application design perfectly because:
1. **Serverless Scale**: It scales to zero when not in use, avoiding active hosting costs.
2. **Docker Native**: It utilizes our existing [Dockerfile](file:///d:/codingProject/GITHUB/Kaggle/Dockerfile) to build the environment, ensuring the MCP server and ADK are run in a controlled environment.
3. **ReadOnly Tolerant**: It runs a read-only filesystem with a writable `/tmp` space. Our logging system handles this natively (catching `OSError` 30 write failures and falling back to memory/console streams).

---

### 8.1 Deployment Steps & Shell Commands (Step-by-Step)

Here are the step-by-step shell commands. Every argument and parameter is annotated with comments explaining its functionality.

```bash
# ==============================================================================
# STEP 1: AUTHENTICATE WITH GOOGLE CLOUD
# ==============================================================================
# This command launches a browser window to authenticate your terminal session
# with your Google Cloud account credentials.
gcloud auth login

# ==============================================================================
# STEP 2: CONFIGURE THE ACTIVE PROJECT ID
# ==============================================================================
# Replace 'YOUR_PROJECT_ID' with your Google Cloud Console project ID.
# This ensures all subsequent commands target the correct cloud resource group.
gcloud config set project YOUR_PROJECT_ID

# ==============================================================================
# STEP 3: ENABLE GCP SERVICES & APIS
# ==============================================================================
# We must enable three core APIs on our project:
# - run.googleapis.com: The Cloud Run service to host and run our container.
# - artifactregistry.googleapis.com: To store built container images.
# - cloudbuild.googleapis.com: To build the container image directly in the cloud.
gcloud services enable \
  run.googleapis.com \
  artifactregistry.googleapis.com \
  cloudbuild.googleapis.com

# ==============================================================================
# STEP 4: BUILD AND DEPLOY TO CLOUD RUN
# ==============================================================================
# This command bundles the local directory, uploads it to Cloud Build,
# packages it into a Docker image, registers it in Artifact Registry, and deploys
# the container to a serverless instance.
#
# Flags explained:
# --source .: Bundles the current directory (excludes files in .dockerignore).
# --region us-central1: The geographic location for hosting the server resources.
# --allow-unauthenticated: Exposes a public URL that anyone can access (no OAuth required).
# --set-env-vars: Passes environment configuration directly to the container.
#                 * GOOGLE_API_KEY: Set your real Gemini API key here.
#                 * TRIPFORGE_WEB_MODE: Instructs Flask to boot in full web interactive mode.
gcloud run deploy tripforge \
  --source . \
  --region us-central1 \
  --allow-unauthenticated \
  --set-env-vars GOOGLE_API_KEY=your_gemini_api_key_here,TRIPFORGE_WEB_MODE=true

# ==============================================================================
# STEP 5: VERIFY LIVE CONTAINER LOGS (OPTIONAL)
# ==============================================================================
# To watch the multi-agent execution output or debug connection errors in real-time,
# run this tail command to stream container logs to your local shell:
gcloud run services logs tail tripforge --region us-central1
```

---

### 8.2 Interactive Environment Checker
Run the code cell below to verify if `gcloud` is installed on your current environment, inspect your active project settings, and print custom commands tailored to your GCP configurations.

In [ ]:
import subprocess
import shutil
import sys

def check_deployment_env():
    print("🔍 Checking Google Cloud CLI environment...")
    gcloud_path = shutil.which("gcloud")
    if not gcloud_path:
        print("❌ 'gcloud' command-line tool not found on PATH.")
        print("💡 Please install Google Cloud SDK: https://cloud.google.com/sdk/docs/install")
        return
    
    print(f"✅ Found Google Cloud SDK at: {gcloud_path}")
    
    try:
        # Check active account
        account_proc = subprocess.run(["gcloud", "config", "get-value", "account"], capture_output=True, text=True, check=True)
        account = account_proc.stdout.strip()
        if account:
            print(f"👤 Active GCP Account: {account}")
        else:
            print("⚠️ No active GCP account configured. You may need to run: gcloud auth login")
            
        # Check active project
        project_proc = subprocess.run(["gcloud", "config", "get-value", "project"], capture_output=True, text=True, check=True)
        project = project_proc.stdout.strip()
        if project and project != "(unset)":
            print(f"🗂️ Active GCP Project: {project}")
        else:
            print("⚠️ No active GCP project configured. You may need to run: gcloud config set project YOUR_PROJECT_ID")
            project = "YOUR_PROJECT_ID"
            
    except Exception as e:
        print(f"⚠️ Error querying gcloud config: {e}")
        project = "YOUR_PROJECT_ID"

    # Print custom commands with code comments
    print("\n🚀 Ready to deploy? Here are the exact commands you need:")
    print("="*60)
    print(f"# Step 1: Login to your Google Cloud Account\n# (Opens a browser window for Google OAuth2 authentication)\ngcloud auth login\n")
    print(f"# Step 2: Set your active Google Cloud project\ngcloud config set project {project}\n")
    print(f"# Step 3: Enable the necessary GCP services\n# - artifactregistry: to store your container image\n# - cloudbuild: to compile/build your Docker image in the cloud\n# - run: to host the serverless container instance\ngcloud services enable run.googleapis.com artifactregistry.googleapis.com cloudbuild.googleapis.com\n")
    print(f"# Step 4: Deploy your Flask app container directly to Cloud Run\n# - --source .: Package and send the current folder to Cloud Build\n# - --region: The GCP region to deploy to\n# - --allow-unauthenticated: Makes the service publicly accessible\n# - --set-env-vars: Sets environment keys. NOTE: Replace your Google API Key!\ngcloud run deploy tripforge \")
    print(f"  --source . \")
    print(f"  --region us-central1 \")
    print(f"  --allow-unauthenticated \")
    print(f"  --set-env-vars GOOGLE_API_KEY=YOUR_GEMINI_API_KEY_HERE,PORT=8080,TRIPFORGE_WEB_MODE=true\n")
    print("="*60)

check_deployment_env()


### 8.3 Continuous Deployment via GitHub Integration

For production workloads, it is recommended to set up **Continuous Deployment (CD)** so that every `git push` to your repository automatically builds and redeploys the application. Google Cloud Run integrates directly with GitHub via Cloud Build:

#### Step 1: Push your Code to GitHub
Make sure your TripForge repository is pushed to a GitHub repository:
```bash
# Initialize git (if not already done)
git init

# Stage and commit your files
git add .
git commit -m "feat: initialize TripForge multi-agent concierage"

# Link to your GitHub repository and push
git remote add origin https://github.com/YOUR_GITHUB_USERNAME/tripforge.git
git branch -M main
git push -u origin main
```

#### Step 2: Set up Cloud Run Continuous Deployment
1. Go to the **[Google Cloud Console](https://console.cloud.google.com/)** and navigate to **Cloud Run**.
2. Click **Create Service**.
3. Under *Deployment trigger*, select **Continuously deploy new revisions from a source repository**.
4. Click **Set up with Cloud Build**.

#### Step 3: Connect GitHub & Authorize
1. In the sidebar panel, select **GitHub** as the Repository Provider.
2. If this is your first time, follow the prompts to authenticate and authorize Google Cloud Build to access your GitHub repositories.
3. Select your repository (`YOUR_GITHUB_USERNAME/tripforge`) from the list.
4. Check the consent box and click **Next**.

#### Step 4: Build Configuration
1. **Branch**: Enter `^main$` (or the branch you want to build from).
2. **Build Type**: Select **Dockerfile**.
3. **Source location**: Leave as `/Dockerfile` (this tells Cloud Run to use the root Dockerfile we commented earlier).
4. Click **Save**.

#### Step 5: Configure Environment Variables & Deploy
1. Under *Variables & Secrets*, add the following environment keys:
   * `GOOGLE_API_KEY`: Set this to your Gemini API Key.
   * `TRIPFORGE_WEB_MODE`: `true`
2. Under *Ingress control*, select **Allow all traffic** (to generate a public URL).
3. Select **Allow unauthenticated invocations** if you want your dashboard to be public.
4. Click **Create**.

#### Step 6: Test and Monitor
* Google Cloud Run will automatically spin up a build pipeline. You can watch the build progress in the **Cloud Build** history tab.
* Once complete, you will receive a secure HTTPS URL (e.g. `https://tripforge-xxxxxx.a.run.app`).
* Any subsequent `git push origin main` will trigger a new build and perform a zero-downtime rolling update!

## 🔮 9. Improvements for Future Systems

To scale TripForge into a highly advanced global travel companion, we propose the following improvements:

1. **Dynamic Tool Auto-Discovery**: Allow agents to dynamically request new MCP server links at runtime based on the target city (e.g., dynamically mounting Tokyo Metro API when planning in Japan).
2. **Human-in-the-Loop Interventions**: Insert approval prompts during the ADK execution flow, allowing travelers to reject specific slots or swap out individual days before the full plan is finalized.
3. **Graph-Search Transit Optimization**: Integrate advanced mathematical optimizers (e.g., Google OR-Tools) to design travel paths between activities that minimize carbon footprint and daily transit expenses.
4. **Persistent Vector Store (RAG)**: Connect the ResearchAgent to a vector database (like ChromaDB or Pgvector) to cache past travel plans, making the platform resilient to API rate limits and completely offline-capable.
5. **Zero-Knowledge Privacy Safeguards**: Execute sensitive logistics verification (such as passport validations) using Zero-Knowledge proofs or Multi-Party Computation so that no personal data leaves the traveler's local browser context.

## 🏁 10. Conclusion & Submission Checklist

This notebook demonstrates all key rubric requirements for the Kaggle Vibecoding Agents Capstone:
1. **Agent / Multi-agent system (ADK)**: Four specialized ADK agents coordinated in a pipeline (`ProfileAgent`, `ResearchAgent`, `ItineraryAgent`, `DisruptionAgent`).
2. **MCP Server**: FastMCP server exposing structured tools via standard I/O.
3. **Security Features**: Symmetric encryption, PII redactors, and HMAC signatures.
4. **Deployability**: Vercel read-only fallback logic, Python 3.12 compatibility, and Google Cloud Run configurations.
5. **Agent Skills**: Custom tool invocation, dynamic self-correcting replanning, asynchronous generator streaming, and cryptographic logging controls.